# Real-World Validation

Every notebook up to this point measures performance on `dataset_malwares.csv`, a curated, pre-labelled CSV. That is not the same question as "does this model work on real files a user actually has on their machine". This notebook is where that question is actually answered: it tests the deployed model (trained only on the original, unaugmented dataset, see `06_evaluation.ipynb`) against real `.exe`/`.dll` files and reports the result honestly, whatever it turns out to be.

**A note on what this notebook used to do:** an earlier version of this notebook also augmented the training data with real legitimate files, specifically because this same real-world scan had found a high false-positive rate, and adding those files brought the number down. That step has been removed. Changing training data specifically to fix a weakness found through evaluation is not sound data science practice, per explicit guidance received on this assignment: it produces a model tuned to one test target rather than one that genuinely generalises. The false-positive rate measured below is now reported as a real finding about the training data (a dataset shift problem, the published dataset's benign class does not structurally resemble typical Windows system files), not something this project tries to fix by reshaping the input data. See `06_evaluation.ipynb`'s intro and the report's limitations section for the full reasoning.

This notebook needs real `.exe`/`.dll` files and real folder paths that do not exist in this sandbox, so every code cell below shows the actual step with a placeholder folder path, but does not execute here. Edit the folder path variables and run this notebook locally, in Jupyter, after `06_evaluation.ipynb` has saved `models/classical_pipeline.joblib`.

Each step is its own plain, sequential cell rather than hidden inside a function, some code (like finding `.exe`/`.dll` files in a folder) repeats across checks instead of being shared, so every step stays visible and easy to follow on its own.

Three checks, in order: a **scan** of files assumed all-legitimate (measures false positives only), a **labeled** check against known-malicious and known-benign folders (measures real accuracy/precision/recall), and an **explain** step that uses SHAP to show exactly why any false positives were flagged.

## Setup: Load the Deployed Model

Loads the exact same file `app.py` loads, so every check below tests exactly what a real user of the deployed app would experience, not a separately-trained copy that might quietly differ.

In [1]:
# Standard library
import os
import sys

# Third-party
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pd.set_option("display.max_columns", None)
sys.path.append("../src")

from constants import LABEL_COLUMN
from explain import build_explainer, explain_prediction
from extract_features import extract_pe_features

MODEL_PATH = "../models/classical_pipeline.joblib"
DATA_PATH = "../data/dataset_malwares.csv"

# Matches app.py MALICIOUS_THRESHOLD exactly, so this notebook measures what a
# real user of the deployed app would actually see.
MALICIOUS_THRESHOLD = 0.5

In [2]:
artifact = joblib.load(MODEL_PATH)
pipeline = artifact["pipeline"]
feature_order = artifact["feature_order"]
print(f"Loaded model trained on {len(feature_order)} features: {feature_order}")

FileNotFoundError: [Errno 2] No such file or directory: '../models/classical_pipeline.joblib'

## Check 1: Scan a Folder Assumed All-Legitimate

Points at a real folder of known-legitimate files (e.g. `C:\Windows\System32`) and reports what fraction get wrongly flagged malicious, the false-positive rate. No ground-truth labels needed here since every file in a real Windows System32 folder is legitimate by construction. The last recorded result with this unaugmented pipeline was around 90% false positives, report whatever this run actually measures, honestly, rather than treating a high number as something to fix by changing the training data.

Set the folder to scan and how many files to look at.

In [ ]:
SCAN_FOLDER = r"C:\Windows\System32"  # edit to a real folder on your machine
SCAN_LIMIT = 200

Walk the folder and collect up to `SCAN_LIMIT` real `.exe`/`.dll` file paths.

In [ ]:
scan_paths = []
for dirpath, _dirs, files in os.walk(SCAN_FOLDER):
    for name in files:
        if name.lower().endswith((".exe", ".dll")):
            scan_paths.append(os.path.join(dirpath, name))
            if len(scan_paths) >= SCAN_LIMIT:
                break
    if len(scan_paths) >= SCAN_LIMIT:
        break

print(f"Found {len(scan_paths)} .exe/.dll files under {SCAN_FOLDER} (limit {SCAN_LIMIT}).")

Classify every file found. Errors are recorded, not silently dropped, so the final counts stay honest.

In [ ]:
scan_rows = []
scan_errors = 0
for i, path in enumerate(scan_paths, 1):
    try:
        with open(path, "rb") as f:
            data = f.read()
        features = extract_pe_features(data, feature_order)
        row_df = pd.DataFrame([features], columns=feature_order)
        proba_malicious = pipeline.predict_proba(row_df)[0][1]
        verdict = "malicious" if proba_malicious >= MALICIOUS_THRESHOLD else "benign"
        scan_rows.append({"path": path, "verdict": verdict, "confidence": round(float(proba_malicious), 4)})
    except Exception as e:
        scan_errors += 1
        scan_rows.append({"path": path, "verdict": "ERROR", "confidence": str(e)[:80]})
    if i % 25 == 0:
        print(f"  ...{i}/{len(scan_paths)} scanned")

scan_results = pd.DataFrame(scan_rows)
scan_results.to_csv("benign_scan_results.csv", index=False)

Report the false-positive rate.

In [ ]:
scan_clean = scan_results[scan_results["verdict"] != "ERROR"]
scan_flagged = scan_clean[scan_clean["verdict"] == "malicious"]
print(f"Scanned {len(scan_clean)} real files successfully ({scan_errors} could not be parsed, skipped).")
print(f"False positives: {len(scan_flagged)} / {len(scan_clean)} "
      f"({len(scan_flagged) / max(len(scan_clean), 1):.1%})")

## Check 2: Score Known-Malicious and Known-Benign Folders

Check 1 above can only measure false positives, since every file it sees is assumed benign. This check scores two folders you provide, one of confirmed-malicious samples and one of confirmed-legitimate software, giving genuine accuracy, precision, and recall against real ground truth, including malicious recall.

The folder-existence check below is deliberate: an earlier version of this tool had none, and would crash with an unclear low-level error on a bad path instead of a clear message.

Set the two labelled folders and how many files to look at in each.

In [ ]:
MALICIOUS_FOLDER = r"C:\path\to\known_malware"  # edit to a real folder on your machine
BENIGN_FOLDER = r"C:\path\to\known_legit"  # edit to a real folder on your machine
LABELED_LIMIT = 200

for folder_label, folder in [("MALICIOUS_FOLDER", MALICIOUS_FOLDER), ("BENIGN_FOLDER", BENIGN_FOLDER)]:
    if not os.path.isdir(folder):
        raise SystemExit(
            f"{folder_label} \"{folder}\" is not a real folder on this machine. "
            f"Replace it with an actual path, not placeholder text."
        )

Find `.exe`/`.dll` files in the malicious folder.

In [ ]:
malicious_paths = []
for dirpath, _dirs, files in os.walk(MALICIOUS_FOLDER):
    for name in files:
        if name.lower().endswith((".exe", ".dll")):
            malicious_paths.append(os.path.join(dirpath, name))
            if len(malicious_paths) >= LABELED_LIMIT:
                break
    if len(malicious_paths) >= LABELED_LIMIT:
        break
print(f"Found {len(malicious_paths)} .exe/.dll files under {MALICIOUS_FOLDER}.")

Find `.exe`/`.dll` files in the benign folder.

In [ ]:
benign_paths = []
for dirpath, _dirs, files in os.walk(BENIGN_FOLDER):
    for name in files:
        if name.lower().endswith((".exe", ".dll")):
            benign_paths.append(os.path.join(dirpath, name))
            if len(benign_paths) >= LABELED_LIMIT:
                break
    if len(benign_paths) >= LABELED_LIMIT:
        break
print(f"Found {len(benign_paths)} .exe/.dll files under {BENIGN_FOLDER}.")

Both folders should have at least one file before continuing, otherwise the results below would be misleadingly empty rather than a real check.

In [ ]:
if not malicious_paths or not benign_paths:
    raise SystemExit(
        "No .exe/.dll files found in one or both folders (0 found above). "
        "Check the paths are correct and actually contain .exe/.dll files."
    )

Classify every file in both folders, tagging each with its asserted true label (1 = malicious, 0 = benign).

In [ ]:
labeled_rows = []
for path in malicious_paths:
    try:
        with open(path, "rb") as f:
            data = f.read()
        feature_values = extract_pe_features(data, feature_order)
        row_df = pd.DataFrame([feature_values], columns=feature_order)
        proba_malicious = pipeline.predict_proba(row_df)[0][1]
        labeled_rows.append({
            "path": path, "true_label": 1,
            "pred_label": 1 if proba_malicious >= MALICIOUS_THRESHOLD else 0,
            "confidence_malicious": round(float(proba_malicious), 4),
        })
    except Exception as e:
        labeled_rows.append({"path": path, "true_label": 1, "pred_label": "ERROR",
                              "confidence_malicious": str(e)[:80]})

for path in benign_paths:
    try:
        with open(path, "rb") as f:
            data = f.read()
        feature_values = extract_pe_features(data, feature_order)
        row_df = pd.DataFrame([feature_values], columns=feature_order)
        proba_malicious = pipeline.predict_proba(row_df)[0][1]
        labeled_rows.append({
            "path": path, "true_label": 0,
            "pred_label": 1 if proba_malicious >= MALICIOUS_THRESHOLD else 0,
            "confidence_malicious": round(float(proba_malicious), 4),
        })
    except Exception as e:
        labeled_rows.append({"path": path, "true_label": 0, "pred_label": "ERROR",
                              "confidence_malicious": str(e)[:80]})

labeled_results = pd.DataFrame(labeled_rows)
labeled_results.to_csv("labeled_test_results.csv", index=False)
print(f"Scored {len(malicious_paths)} malicious + {len(benign_paths)} benign files.")

Report accuracy, precision, recall, and ROC-AUC against the real ground truth.

In [ ]:
labeled_clean = labeled_results[labeled_results["pred_label"] != "ERROR"].copy()
labeled_errors = len(labeled_results) - len(labeled_clean)
if labeled_errors:
    print(f"({labeled_errors} files could not be parsed, skipped from scoring.)")

y_true = labeled_clean["true_label"].astype(int)
y_pred = labeled_clean["pred_label"].astype(int)
print(f"--- Results on {len(labeled_clean)} real, known-labelled files ---")
print(classification_report(y_true, y_pred, target_names=["benign", "malicious"]))
print("Confusion matrix (rows=actual, cols=predicted, order=[benign, malicious]):")
print(confusion_matrix(y_true, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_true, labeled_clean["confidence_malicious"].astype(float)), 4))

## Check 3: Explain False Positives with SHAP

Takes Check 1's results, picks the highest-confidence false positives, and shows a SHAP breakdown for each: which features pushed it toward malicious, compared against the typical benign and malicious value for that feature in training. A value far outside *both* training averages usually means a train/serve skew (the model never saw a real value like this during training), not that the value is inherently suspicious, this is exactly how the original 7 `DROPPED_FEATURES` zero-variance bug was traced back in `01_data_understanding.ipynb`.

Pick the highest-confidence false positives from Check 1.

In [ ]:
TOP_N = 5
flagged = scan_results[scan_results["verdict"] == "malicious"].sort_values("confidence", ascending=False)
top_false_positives = flagged.head(TOP_N)
print(f"{len(flagged)} false positives found, explaining the top {len(top_false_positives)}.")

Compute the per-feature training averages (benign vs malicious) to compare each false positive against.

In [ ]:
train_df = pd.read_csv(DATA_PATH)
feature_stats = train_df.groupby("Malware")[feature_order].agg(["mean"])
explainer = build_explainer(pipeline)

Explain each false positive: which features pushed the model toward malicious, and how each compares to the training averages.

In [ ]:
for _, r in top_false_positives.iterrows():
    path = r["path"]
    print("=" * 100)
    print(f"{path}  (model confidence malicious: {r['confidence']:.0%})")
    try:
        with open(path, "rb") as f:
            data = f.read()
        feature_values = extract_pe_features(data, feature_order)
        features = dict(zip(feature_order, feature_values))
    except Exception as e:
        print(f"  Could not re-extract features: {e}")
        continue

    row_df = pd.DataFrame([feature_values], columns=feature_order)
    toward_malicious, _ = explain_prediction(explainer, pipeline, row_df.values, feature_order, n=8)

    print(f"  {'feature':<28}{'this file':>14}{'shap':>10}{'benign mean':>16}{'malicious mean':>16}")
    for name, shap_val in toward_malicious:
        this_val = features[name]
        benign_mean = feature_stats.loc[0, (name, "mean")]
        malicious_mean = feature_stats.loc[1, (name, "mean")]
        print(f"  {name:<28}{this_val:>14.0f}{shap_val:>10.3f}{benign_mean:>16.1f}{malicious_mean:>16.1f}")
    print()

## Summary

- Loads the exact deployed pipeline `app.py` uses (`models/classical_pipeline.joblib`, `CORE_TRAITS`, trained on the original unaugmented `dataset_malwares.csv`, `MALICIOUS_THRESHOLD = 0.5`), so every check here reflects what a real user of the app would actually experience.
- Every step above is a plain, sequential cell rather than a function, folder paths are set as plain variables at the top of each check and edited directly, no reusable wrapper hides what each step does.
- **Check 1 (scan)**: false-positive rate against a folder assumed all-legitimate. The last recorded result for this unaugmented pipeline was around 90% false positives against real `C:\Windows\System32` files, understood as a dataset shift problem, not a bug, and reported honestly rather than fixed by changing training data. **Re-run this to get a current number** once `06_evaluation.ipynb` has been freshly re-run.
- **Check 2 (labeled)**: real accuracy/precision/recall against known-malicious and known-benign folders, including malicious recall, which Check 1 alone cannot measure. Refuses to run silently on a bad or empty folder path.
- **Check 3 (explain)**: SHAP breakdown of the highest-confidence false positives from Check 1, read against per-feature training averages, useful for understanding *why* the model over-flags certain real files, even though the fix for that is better training data collection (future work), not a patch applied here.
- **To run this notebook for real:** open it in Jupyter on a machine with real `.exe`/`.dll` files available, edit the folder path variables in Checks 1 and 2 to real paths, and re-run after any change to the deployed model in `06_evaluation.ipynb`. Report whatever false-positive rate is found honestly in the project report, alongside the held-out validation numbers, not instead of them.